# Areas, centroids and second moments

The section properties of a profile, its area, centroid and second moments of area, are what a beam calculation
needs. `center` is the centroid in 3D, and `Measure()` adds the second moments about it in the plane of the object:

- `Ih`, the integral of v², about the h axis;
- `Iv`, the integral of h², about the v axis;
- `Ihv`, the product moment, the integral of h v;
- `I1 ≥ I2`, the principal second moments, and `principal_angle`, the angle of the first principal axis in degrees.

In [1]:
import math
import planex as px

In [2]:
from docs_figures import Row, Vertices

## Everything at once

`Measure()` returns every measurement of an object as a dict. The example is an IPE 200 beam section without its root
radii: 200 high and 100 wide, with flanges 8.5 thick and a web 5.6 thick (Figure 1, with the centroid marked).

In [3]:
flange = px.QuickSketch().RectangleC(100, 8.5).Face()
web = px.QuickSketch().RectangleC(5.6, 200 - 2 * 8.5).Face()
beam = px.Unify([flange.Translated((0, 95.75)), web, flange.Translated((0, -95.75))])

In [4]:
Row((beam, "An IPE 200 and its center", {"marks": [beam.center[:2]]}), caption="An IPE 200 section")

In [5]:
m = beam.Measure()
for key in ["type", "area", "center", "Ih", "Iv", "I1", "I2", "principal_angle", "bounding_box"]:
    print(key, m[key])

type face
area 2724.799999999999
center (-3.337840214962303e-16, -2.002704128977382e-15, 0.0)
Ih 18455902.266666662
Iv 1419344.8106666664
I1 18455902.266666662
I2 1419344.8106666654
principal_angle 6.125728674641656e-33
bounding_box ((-50.0, -100.0), (50.0, 100.0))


## Two sections

The first example is a T-slot nut, a flange of 18 × 4 below a neck of 10 × 6, with its centroid marked (Figure 2a).
Its half is drawn and mirrored. The centroid of two rectangles lies at v = (A1 v1 + A2 v2) / (A1 + A2).

In [6]:
half = px.QuickSketch().Line(9).Line(0, 4).Line(-4, 0).Line(0, 6).Line((0, 10)).Close().Face()
nut = half + half.Mirrored((0, 0), (0, 1))
A1, A2 = 18 * 4, 10 * 6
print("nut centroid", nut.center, "exact v", (A1 * 2 + A2 * (4 + 3)) / (A1 + A2))

nut centroid (1.0765799026668185e-16, 4.2727272727272725, 0.0) exact v 4.2727272727272725


The second example is a sharp-cornered equal angle of 40 × 40 × 5, with its centroid and its principal axes dashed
(Figure 2b). The principal axes lie at 45° to the legs. Its second moments follow from the two rectangles about the
outer corner, moved to the centroid. With L = 40 and t = 5:

In [7]:
L, t = 40, 5
angle = px.QuickSketch().Line(L).Line(0, t).Line(t - L, 0).Line(0, L - t).Line(-t, 0).Close().Face()
m = angle.Measure()

A = t * (2 * L - t)
c = (L**2 + L * t - t**2) / (2 * (2 * L - t))
I = (t * L**3 + (L - t) * t**3) / 3 - A * c**2
Ihv = t**2 * (2 * L**2 - t**2) / 4 - A * c**2
for key, exact in {"area": A, "Ih": I, "Iv": I, "I1": I + abs(Ihv), "I2": I - abs(Ihv)}.items():
    print(key, m[key], "exact", exact)
print("principal angle", m["principal_angle"])

area 375.0 exact 375
Ih 55614.583333333336 exact 55614.58333333333
Iv 55614.583333333336 exact 55614.58333333333
I1 88281.25 exact 88281.25
I2 22947.91666666667 exact 22947.916666666657
principal angle 45.0


In [8]:
cu, cv = angle.center[:2]
k = math.radians(m["principal_angle"])
axes = [px.Line((cu - 30 * math.cos(q), cv - 30 * math.sin(q)), (cu + 30 * math.cos(q), cv + 30 * math.sin(q)))
        for q in (k, k + math.pi / 2)]
Row((nut, "A T-slot nut and its centroid", {"marks": [nut.center[:2]]}),
    (angle, "An equal angle and its principal axes", {"marks": [(cu, cv)], "dashed": axes, "view": ((-8, 48), (-8, 48))}),
    caption="Centroids and principal axes")